In [66]:
# Start with the CFS (2017)

cfs_path = '/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/CFS/'
cfs_mapping_path = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/Mapping Files/CFS_sector_aggregation.xlsx"

In [64]:
len(df)

5978523

,naics2012,naics2012_ttl,mapping,CDP order
0,423,"Merchant wholesalers, durable goods",Wholesale and Retail Trade,13.0
1,333,Machinery manufacturing,Machinery,9.0
2,4239,Miscellaneous durable goods merchant wholesalers,Wholesale and Retail Trade,13.0
3,4236,Household appliances and electrical and electr...,Wholesale and Retail Trade,13.0
4,337,Furniture and related product manufacturing,"Furniture and Related Products, and Miscellane...",12.0
5,4237,"Hardware, plumbing and heating equipment and s...",Wholesale and Retail Trade,13.0
6,324,Petroleum and coal products manufacturing,Petroleum and Coal Products,4.0
7,339,Miscellaneous manufacturing,"Furniture and Related Products, and Miscellane...",12.0
8,4249,Miscellaneous nondurable goods merchant wholes...,Wholesale and Retail Trade,13.0
9,325,Chemical manufacturing,Chemical,5.0


# WIOD 2014

In [ ]:
wiod_path = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/WIOD/WIOT2014_Nov16_ROW.xlsb"
wiod_mapping_sector_path = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/Mapping Files/WIOD_sector_aggregation.xlsx"
out_folder = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/X_in_jk/"

In [6]:
%pip install pyxlsb

  Using cached pyxlsb-1.0.10-py2.py3-none-any.whl.metadata (2.5 kB)
Using cached pyxlsb-1.0.10-py2.py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pyxlsb

In [ ]:
import pandas as pd

# ------------------------------
# 1) Read the matrix from Excel
# ------------------------------
# - skiprows=4 so that the first row we read is row 5 of the sheet
# - header=[0,1] means: the first two rows we read become a 2-level column header
#   (row 5 is level 0: "ImportingCountry", row 6 is level 1: "ImportingIndustry")
# - index_col=[2,3] means: use columns C and D (zero-based: 2 and 3) as the row index
#   (ExportingCountry, ExportingIndustry)
df = pd.read_excel(
    wiod_path,
    sheet_name="2014",
    skiprows=4,        # Adjust if you have more or fewer label rows
    header=[0, 1],     # Row 5 => upper col labels, Row 6 => lower col labels
    index_col=[2, 3],  # Column C => ExportingCountry, Column D => ExportingIndustry
    # usecols="C:ZZ",  # Optionally restrict columns if needed
)

# By default, pandas will name the two column levels something like (None, None).
# We can rename them:
df.columns.names = ["ImportingCountry", "ImportingIndustry"]
df.index.names = ["ExportingCountry", "ExportingIndustry"]

# df is now a 2D table whose:
#   - row index = (ExportingCountry, ExportingIndustry)
#   - columns   = (ImportingCountry, ImportingIndustry)
#   - cell values = trade flows (or I/O flows).

# ------------------------------
# 2) Reshape from wide to long
# ------------------------------
# We "stack" over the two column levels:
df_long = df.stack(level=[0, 1]).reset_index()

# The result has columns:
#   ["ExportingCountry", "ExportingIndustry", "ImportingCountry", "ImportingIndustry", 0]
# Rename the last one to "Value" (or something relevant).
df_long.columns = [
    "ExportingCountry",
    "ExportingIndustry",
    "ImportingCountry",
    "ImportingIndustry",
    "Value",
]

# ------------------------------
# 3) Filter out invalid industries (optional)
# ------------------------------
# If valid industries are r1..r56 for exporting and c1..c56 for importing:
valid_r = [f"r{i}" for i in range(1, 57)]
valid_c = [f"c{i}" for i in range(1, 57)]

df_long = df_long[
    df_long["ExportingIndustry"].isin(valid_r)
    & df_long["ImportingIndustry"].isin(valid_c)
]

# ------------------------------
# 4) Collapse certain countries
# ------------------------------
# E.g. CHE, HRV, LUX, LVA, MLT, NOR => "ROW"
row_countries = ["CHE", "HRV", "LUX", "LVA", "MLT", "NOR"]

df_long["ExportingCountry"] = df_long["ExportingCountry"].where(
    ~df_long["ExportingCountry"].isin(row_countries), 
    "ROW"
)
df_long["ImportingCountry"] = df_long["ImportingCountry"].where(
    ~df_long["ImportingCountry"].isin(row_countries), 
    "ROW"
)

# ------------------------------
# 5) Summation / Aggregation
# ------------------------------
df_agg = df_long.groupby(
    ["ExportingCountry", "ExportingIndustry", 
     "ImportingCountry", "ImportingIndustry"],
    as_index=False
)["Value"].sum()


print(df_agg.head(15))


/var/folders/g9/ggrmf_5j6tx4rv5qdhs0slq40000gn/T/ipykernel_29984/3384075620.py:34: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_long = df.stack(level=[0, 1]).reset_index()


   ExportingCountry ExportingIndustry ImportingCountry ImportingIndustry  \
0               AUS                r1              AUS                c1   
1               AUS                r1              AUS               c10   
2               AUS                r1              AUS               c11   
3               AUS                r1              AUS               c12   
4               AUS                r1              AUS               c13   
5               AUS                r1              AUS               c14   
6               AUS                r1              AUS               c15   
7               AUS                r1              AUS               c16   
8               AUS                r1              AUS               c17   
9               AUS                r1              AUS               c18   
10              AUS                r1              AUS               c19   
11              AUS                r1              AUS                c2   
12          

In [12]:
len(df_agg.ImportingCountry.unique())

38

In [ ]:
# For example, save to CSV
df_agg.to_csv(out_folder+ "long_fix_countries.csv", index=False)


In [29]:
df_agg = pd.read_csv(out_folder + "long_fix_countries.csv")

In [30]:
df_agg

,ExportingCountry,ExportingIndustry,ImportingCountry,ImportingIndustry,Value
0,AUS,r1,AUS,c1,12924.179691
1,AUS,r1,AUS,c10,0.226825
2,AUS,r1,AUS,c11,86.721589
3,AUS,r1,AUS,c12,109.147573
4,AUS,r1,AUS,c13,147.252719
...,...,...,...,...,...
4528379,USA,r9,USA,c56,0.000000
4528380,USA,r9,USA,c6,743.552862
4528381,USA,r9,USA,c7,75.372114
4528382,USA,r9,USA,c8,208.748822


In [34]:
mapping = pd.read_excel(wiod_mapping_sector_path)

In [35]:
mapping

,CDP 2019,Order in CDP 2019,OrderExcldNon,WIOD 2016,output_code,input_code,Notes
0,"Wood Products, Paper, Printing, and Related Su...",4.0,3.0,Printing and reproduction of recorded media,r9,c9,NaN
1,"Wood Products, Paper, Printing, and Related Su...",4.0,3.0,Manufacture of paper and paper products,r8,c8,NaN
2,"Wood Products, Paper, Printing, and Related Su...",4.0,3.0,Manufacture of wood and of products of wood an...,r7,c7,NaN
3,"Textile, Textile Product Mills, Apparel, Leath...",3.0,2.0,"Manufacture of textiles, wearing apparel and l...",r6,c6,NaN
4,NaN,NaN,NaN,Activities of extraterritorial organizations a...,r56,c56,drop I think.
5,Non Employment sector,1.0,NaN,Activities of households as employers; undiffe...,r55,c55,NaN
6,Other Services,23.0,22.0,Other service activities,r54,c54,NaN
7,Health Care,21.0,20.0,Human health and social work activities,r53,c53,NaN
8,Education,20.0,19.0,Education,r52,c52,NaN
9,Other Services,23.0,22.0,Public administration and defence; compulsory ...,r51,c51,?? Other services??


In [36]:
%pip install openpyxl
import openpyxl

Note: you may need to restart the kernel to use updated packages.


In [41]:
# 1) Build lookup dictionaries from your ‘mapping’ dataframe
out_map = mapping.set_index('output_code')['OrderExcldNon'].to_dict()
in_map  = mapping.set_index('input_code')['OrderExcldNon'].to_dict()
df_agg_sector_fix = df_agg.copy()
# 2) Map the old codes in df_agg to the new “Order in CDP 2019”
df_agg_sector_fix['ExportingIndustry'] = df_agg_sector_fix['ExportingIndustry'].map(out_map)
df_agg_sector_fix['ImportingIndustry'] = df_agg_sector_fix['ImportingIndustry'].map(in_map)

# 3) Drop rows where either mapped industry is NaN
df_agg_sector_fix = df_agg_sector_fix.dropna(subset=['ExportingIndustry','ImportingIndustry'])

# 4) Collapse (aggregate) non-unique rows, for example by summing 'Value'
df_agg_sector_fix = df_agg_sector_fix.groupby(
    ['ExportingCountry','ExportingIndustry','ImportingCountry','ImportingIndustry'],
    as_index=False
)['Value'].sum()


In [42]:
# For example, save to CSV
df_agg_sector_fix.to_csv(out_folder+ "long_fix_countries_and_sectors.csv", index=False)


In [43]:
df_agg_sector_fix

,ExportingCountry,ExportingIndustry,ImportingCountry,ImportingIndustry,Value
0,AUS,1.0,AUS,1.0,10851.759013
1,AUS,1.0,AUS,2.0,262.228559
2,AUS,1.0,AUS,3.0,40.510798
3,AUS,1.0,AUS,4.0,2.897519
4,AUS,1.0,AUS,5.0,352.067899
...,...,...,...,...,...
698891,USA,22.0,USA,18.0,214761.185588
698892,USA,22.0,USA,19.0,25490.798838
698893,USA,22.0,USA,20.0,241208.476296
698894,USA,22.0,USA,21.0,116928.130124


In [70]:
df_agg_sector_fix

,ExportingCountry,ExportingIndustry,ImportingCountry,ImportingIndustry,Value
0,AUS,1.0,AUS,1.0,10851.759013
1,AUS,1.0,AUS,2.0,262.228559
2,AUS,1.0,AUS,3.0,40.510798
3,AUS,1.0,AUS,4.0,2.897519
4,AUS,1.0,AUS,5.0,352.067899
...,...,...,...,...,...
698891,USA,22.0,USA,18.0,214761.185588
698892,USA,22.0,USA,19.0,25490.798838
698893,USA,22.0,USA,20.0,241208.476296
698894,USA,22.0,USA,21.0,116928.130124


In [71]:

domestic_usa_trade = df_agg_sector_fix[
    (df_agg_sector_fix['ExportingCountry'] == 'USA') & 
    (df_agg_sector_fix['ImportingCountry'] == 'USA')
]

# Group by exporting industry and sum Value
domestic_expenditure = domestic_usa_trade.groupby('ExportingIndustry', as_index=False)['Value'].sum()


In [72]:
domestic_expenditure

,ExportingIndustry,Value
0,1.0,3.702758e+05
1,2.0,5.021600e+04
2,3.0,2.875896e+05
3,4.0,3.924727e+05
4,5.0,3.776814e+05
5,6.0,1.617610e+05
6,7.0,9.119060e+04
7,8.0,5.297387e+05
8,9.0,1.214166e+05
9,10.0,2.010416e+05


In [171]:
import pandas as pd

# Replace 'your_file.dta' with the path to your .dta file
cfs_2017 = pd.read_stata(cfs_path + 'states_ind_bilateral_trade2017.dta')
cfs_2017 = cfs_2017[cfs_2017['origin_abrv'] != 'USA']
cfs_2017 = cfs_2017[cfs_2017['origin_abrv'] != 'DC']

cfs_2017 = cfs_2017[cfs_2017['dest_abrv'] != 'USA']
cfs_2017 = cfs_2017[cfs_2017['dest_abrv'] != 'DC']
# Display the first few rows of the dataframe
cfs_2017.head()

cfs_2017.to_csv(cfs_path + 'states_ind_bilateral_trade2017.csv')

In [172]:
cfs_2017 = cfs_2017[['dest_abrv', 'origin_abrv', 'naics2012', 'val']]

# some summary stats

In [173]:
# Group by naics2012, sum the val column, and sort in descending order
cfs_2017 = cfs_2017[cfs_2017['naics2012'] != '00']
result = cfs_2017.groupby('naics2012')['val'].sum().sort_values(ascending=False)


In [177]:
result.sum()

np.int64(31467702)

In [178]:
cfs_mapping = pd.read_excel(cfs_mapping_path)


In [179]:
cfs_mapping['naics2012'] = cfs_mapping['naics2012'].astype(str)
cfs_2017['naics2012'] = cfs_2017['naics2012'].astype(str)

# Create a mapping dictionary from cfs_mapping
mapping_dict = dict(zip(cfs_mapping['naics2012'], cfs_mapping['CDP order']))

# Map the 'naics_2012' column in cfs_2017 to a new 'CDP order' column
cfs_2017['CDP order'] = cfs_2017['naics2012'].map(mapping_dict)


In [180]:
cfs_2017_collapsed = cfs_2017.groupby(['dest_abrv', 'origin_abrv', 'CDP order'], as_index=False)['val'].sum()

In [181]:
cfs_2017['naics2012'].unique()

array(['423', '333', '4239', '4236', '337', '4237', '324', '339', '4249',
       '325', '4541', '42', '326', '424', '321', '4241', '335', '4243',
       '312', '4233', '4234', '316', '313', '334', '4232', '4244', '331',
       '336', '4245', '4931', '4231', '31-33', '5111', '327', '332',
       '315', '4242', '323', '4238', '4247', '551114', '4235', '212',
       '311', '4248', '322', '45431', '4246', '314'], dtype=object)

In [182]:
cfs_2017_collapsed

,dest_abrv,origin_abrv,CDP order,val
0,AK,AK,1.0,2021
1,AK,AK,2.0,9
2,AK,AK,3.0,142
3,AK,AK,4.0,1990
4,AK,AK,5.0,41
...,...,...,...,...
28965,WY,WY,9.0,72
28966,WY,WY,10.0,4
28967,WY,WY,11.0,0
28968,WY,WY,12.0,4406


# Get each state's share of [that sector's expenditure]

In [183]:
cfs_2017_collapsed.columns

Index(['dest_abrv', 'origin_abrv', 'CDP order', 'val'], dtype='object')

In [184]:
import pandas as pd

# Suppose your DataFrame is called df and has columns:
#   ['dest_abbrv', 'origin_abbrv', 'CDP order', 'val']

# 1) Sum over all origins to get total spending by (CDP order, dest_abbrv).
df_sum = cfs_2017_collapsed.groupby(['CDP order', 'dest_abrv'], as_index=False)['val'].sum()

# 2) Compute total expenditures by each sector (CDP order).
df_total = df_sum.groupby('CDP order', as_index=False)['val'].sum()
df_total = df_total.rename(columns={'val': 'sector_total'})

# 3) Merge back to compute shares
df_shares = pd.merge(df_sum, df_total, on='CDP order', how='left')
df_shares['share'] = df_shares['val'] / df_shares['sector_total']

# df_shares now has columns:
#   ['CDP order', 'dest_abbrv', 'val', 'sector_total', 'share']
# where share is the fraction of sector expenditure going to each state.


In [185]:
df_shares

,CDP order,dest_abrv,val,sector_total,share
0,1.0,AK,2131,1071314,0.001989
1,1.0,AL,11678,1071314,0.010901
2,1.0,AR,16980,1071314,0.015850
3,1.0,AZ,18048,1071314,0.016847
4,1.0,CA,131247,1071314,0.122510
...,...,...,...,...,...
645,13.0,VT,30711,19502236,0.001575
646,13.0,WA,438802,19502236,0.022500
647,13.0,WI,371530,19502236,0.019051
648,13.0,WV,84953,19502236,0.004356


# Bilateral expenditure shares 
# across US states [no sectors?]

In [186]:
import pandas as pd
import numpy as np

# Example: assume your DataFrame is called cfs_2017_collapsed
df = cfs_2017_collapsed

# Step 1: (Optional) group if you have multiple rows per (origin, destination).
#         If each pair is already unique, you can skip this groupby.
df_agg = df.groupby(['origin_abrv', 'dest_abrv'], as_index=False)['val'].sum()

# Step 2: pivot so that rows=origins, columns=destinations
pivot_df = df_agg.pivot_table(
    index='origin_abrv',
    columns='dest_abrv',
    values='val',
    aggfunc='sum',      # 'sum' if not already unique
    fill_value=0
)

# Step 3: create shares so that each column sums to 1
# (i.e., share of each destination's total spending on each origin)
pivot_shares = pivot_df.div(pivot_df.sum(axis=0), axis=1)

# Step 4: convert to a 2D NumPy array
result_matrix = pivot_shares.to_numpy()

print(result_matrix)


[[6.18893720e-01 0.00000000e+00 0.00000000e+00 ... 3.20925292e-06
  0.00000000e+00 0.00000000e+00]
 [5.87613116e-05 4.85346909e-01 8.35686400e-03 ... 4.19770282e-03
  3.50073037e-03 9.05739863e-04]
 [1.95871039e-05 5.23324693e-03 4.38539942e-01 ... 2.63800590e-03
  5.70862506e-04 2.02211690e-03]
 ...
 [1.44944568e-03 7.00141632e-03 9.41519518e-03 ... 4.44407716e-01
  6.49775853e-03 3.60189573e-03]
 [1.95871039e-05 3.60760001e-04 6.14798192e-04 ... 7.52569809e-04
  3.79707517e-01 2.94892048e-04]
 [1.95871039e-05 1.78153087e-05 8.78283132e-06 ... 1.42811755e-04
  8.39503685e-06 4.39957873e-01]]


In [187]:
result_matrix
column_sums = result_matrix.sum(axis=0)
print(column_sums)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1.]


In [188]:
pivot_df

dest_abrv,AK,AL,AR,AZ,CA,CO,CT,DE,FL,GA,...,SD,TN,TX,UT,VA,VT,WA,WI,WV,WY
origin_abrv,,,,,,,,,,,,,,,,,,,,,
AK,31597,0,0,7,128,0,0,0,10,0,...,0,0,40,0,4,2,3685,2,0,0
AL,3,217946,1903,1772,14537,1136,378,451,31855,44899,...,95,28013,22250,764,3004,19,1332,2616,417,43
AR,1,2350,99863,1544,6723,806,205,31,2887,4581,...,94,8759,20410,977,840,1,972,1644,68,96
AZ,39,1113,209,204378,52855,2963,1337,33,5338,1222,...,82,1840,21866,2375,2085,7,3612,1324,11,251
CA,1670,19160,5794,80930,2498320,27289,12412,1864,86820,37250,...,1221,28676,160856,34744,25544,788,71057,18527,1880,1534
CO,89,793,254,3344,22062,224871,476,39,4250,1814,...,1762,1768,14396,12120,1086,0,2993,1729,51,9799
CT,39,714,138,1250,10151,894,164939,195,6907,2996,...,23,11996,7655,610,2405,2032,1035,1685,104,18
DE,0,104,95,1809,2846,418,699,31710,462,655,...,6,1247,1791,73,1068,52,341,417,17,5
FL,25,15227,1537,3776,24070,2091,3495,681,824388,37943,...,201,8685,26301,772,10189,168,6020,4024,537,126
